# Naive Bayes Classification – Play Tennis Dataset

This notebook applies **Categorical Naive Bayes** to predict whether tennis will be played (`Yes` or `No`).

**Note:** Keep `play_tennis.csv` in the same folder as this notebook before running it. The `day` column is an identifier, so it is not used as a feature.

In [1]:
# Import required libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


## 1. Load and explore the dataset

In [2]:
# Load the dataset
df = pd.read_csv("play_tennis.csv")

# Display the first few rows and dataset information
display(df.head())
print("Dataset shape:", df.shape)
print("\\nMissing values:")
print(df.isnull().sum())


,day,outlook,temp,humidity,wind,play
0,D1,Sunny,Hot,High,Weak,No
1,D2,Sunny,Hot,High,Strong,No
2,D3,Overcast,Hot,High,Weak,Yes
3,D4,Rain,Mild,High,Weak,Yes
4,D5,Rain,Cool,Normal,Weak,Yes


Dataset shape: (14, 6)
\nMissing values:
day         0
outlook     0
temp        0
humidity    0
wind        0
play        0
dtype: int64


## 2. Prepare the data
The `day` column is removed because it is only a record identifier. The remaining categorical input columns are encoded as non-negative integers, as required by `CategoricalNB`.

In [3]:
# Separate input features and target
X = df.drop(columns=["day", "play"])
y = df["play"]

# Encode categorical input features
feature_encoder = OrdinalEncoder()
X_encoded = feature_encoder.fit_transform(X).astype(int)

# Encode target labels (No/Yes)
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

print("Feature columns:", X.columns.tolist())
print("Target classes:", target_encoder.classes_)
print("\\nEncoded feature sample:")
display(pd.DataFrame(X_encoded, columns=X.columns).head())


Feature columns: ['outlook', 'temp', 'humidity', 'wind']
Target classes: ['No' 'Yes']
\nEncoded feature sample:


,outlook,temp,humidity,wind
0,2,1,0,1
1,2,1,0,0
2,0,1,0,1
3,1,2,0,1
4,1,0,1,1


## 3. Split the dataset

In [4]:
# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y_encoded,
    test_size=0.30,
    random_state=42,
    stratify=y_encoded
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 9
Testing samples: 5


## 4. Train the Naive Bayes model

In [5]:
# Create and train the Categorical Naive Bayes model
model = CategoricalNB()
model.fit(X_train, y_train)

print("Model training completed successfully.")


Model training completed successfully.


## 5. Make predictions and evaluate the model

In [6]:
# Predict the test data
y_pred = model.predict(X_test)

# Convert encoded labels back to Yes/No
actual = target_encoder.inverse_transform(y_test)
predicted = target_encoder.inverse_transform(y_pred)

results = pd.DataFrame({
    "Actual": actual,
    "Predicted": predicted
})
display(results)

print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("\\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_, zero_division=0))


,Actual,Predicted
0,Yes,Yes
1,Yes,Yes
2,No,Yes
3,Yes,Yes
4,No,Yes


Accuracy: 60.0 %
\nConfusion Matrix:
[[0 2]
 [0 3]]
\nClassification Report:
              precision    recall  f1-score   support

          No       0.00      0.00      0.00         2
         Yes       0.60      1.00      0.75         3

    accuracy                           0.60         5
   macro avg       0.30      0.50      0.38         5
weighted avg       0.36      0.60      0.45         5



## 6. Predict for a new weather condition

In [7]:
# Example: Outlook=Sunny, Temp=Cool, Humidity=High, Wind=Strong
new_data = pd.DataFrame([{
    "outlook": "Sunny",
    "temp": "Cool",
    "humidity": "High",
    "wind": "Strong"
}])

# Use the same feature encoder used during training
new_data_encoded = feature_encoder.transform(new_data).astype(int)

# Predict and display the result
new_prediction = model.predict(new_data_encoded)
predicted_class = target_encoder.inverse_transform(new_prediction)

print("Predicted result:", predicted_class[0])


Predicted result: No


## 7. Conclusion

Categorical Naive Bayes was trained on the Play Tennis dataset using `outlook`, `temp`, `humidity`, and `wind` as input features. The model predicts whether the outcome is `Yes` or `No`.

**Algorithm idea:** Naive Bayes uses Bayes' theorem and assumes that the input features are conditionally independent given the target class.